# Fixed effects, clustering, and linear contrasts

A teaching notebook for the Uganda refugee-hosting replication. This uses the real released panel. Original study and data: Zhou, Grossman, and Ge (2023), DOI 10.7910/DVN/TXSZDC. Codex assisted with notebook development.

Learning goals: understand absorption, compare a coefficient with a contrast, and distinguish replication from causal validation.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
root = Path.cwd()
if root.name == 'teaching': root = root.parent
sys.path.insert(0, str(root))
from run_analysis import load_data, estimate, OUTCOMES
from src.econometrics import absorb
raw, complete, breaks = load_data()
print(f'{len(raw):,} rows, {raw.parish_id.nunique():,} parishes')

26,890 rows, 5,378 parishes


## 1. What does absorption remove?

Parish effects account for time-invariant parish characteristics. Region-wave effects account for shared regional shifts. Residualizing the outcome and regressors against these effects lets OLS estimate the remaining relationship. This computational transformation does not make settlement location random.

In [2]:
sample = complete[complete.min_distance_01 < 150].copy()
values = sample[['presence']].to_numpy()
residual, iterations = absorb(values, [sample.parish_id, sample.region_year])
check = sample[['parish_id', 'region_year']].copy()
check['residual'] = residual[:, 0]
print('Projection iterations:', iterations)
print('Largest parish residual mean:', check.groupby('parish_id').residual.mean().abs().max())
print('Largest region-wave residual mean:', check.groupby('region_year').residual.mean().abs().max())

Projection iterations: 2
Largest parish residual mean: 1.7763568394002506e-16
Largest region-wave residual mean: 6.341344165257924e-17


## 2. A post-2011 contrast uses covariance

The desired quantity is the 2020 presence interaction minus the 2011 interaction. Its variance is Var(b2020) + Var(b2011) − 2 Cov(b2020,b2011). Adding separate standard errors would be incorrect.

In [3]:
fit, model_sample = estimate(complete, OUTCOMES['Public primary schools'])
contrast = fit.contrast({'presence_2020': 1, 'presence_2011': -1})
print(pd.Series(contrast))

estimate        0.047906
se              0.012984
ci_low          0.022449
ci_high         0.073363
p_value         0.000228
n           18185.000000
clusters     3637.000000
dtype: float64


## 3. Check an external benchmark

S16 is a binary post-period specification. It answers a different question from the continuous slope contrast. Agreement with published rounding demonstrates implementation consistency, not proof of causal identification.

In [4]:
reconciliation = pd.read_csv(root / 'output/s16_reconciliation.csv')
print(reconciliation[['outcome','estimate','published_estimate','se','published_se','n']].to_string(index=False))
assert reconciliation[['coefficient_matches_rounding','se_matches_rounding','n_matches']].all().all()

                 outcome  estimate  published_estimate       se  published_se     n
  Public primary schools  0.057973               0.058 0.024696         0.025 18178
Public secondary schools -0.011653              -0.012 0.028555         0.029 18178
            Road density  0.179052               0.179 0.032295         0.032 10904
             HC2 clinics -0.160318              -0.160 0.033612         0.034 14548
             HC3 centers  0.109709               0.110 0.027379         0.027 14548
             HC4 centers -0.060799              -0.061 0.024913         0.025 14548
           HC5 hospitals  0.068004               0.068 0.027535         0.028 14548
      Public goods index  0.030697               0.031 0.004726         0.005 18178


## 4. Read heterogeneity on two outcome scales

Primary-school standardized outcomes have a different normalization in each wave. Compare the group difference and its interval directly. Do not infer a group difference merely because one group has a small p-value and another does not.

In [5]:
extension = pd.read_csv(root / 'output/baseline_access_extension.csv')
raw_units = pd.read_csv(root / 'output/raw_unit_extension.csv')
print(extension[(extension.outcome == 'Public primary schools') & (extension.post_year == 2020)].to_string(index=False))
print(raw_units[(raw_units.outcome == 'Public primary schools') & (raw_units.post_year == 2020)].to_string(index=False))

               outcome  post_year                           group  estimate       se    ci_low  ci_high  p_value     n  clusters
Public primary schools       2020    Above-median baseline access  0.054605 0.017977  0.019359 0.089850 0.002402 18185      3637
Public primary schools       2020 At/below-median baseline access  0.027306 0.009395  0.008886 0.045726 0.003677 18185      3637
Public primary schools       2020       Difference between groups -0.027299 0.015423 -0.057537 0.002940 0.076808 18185      3637
               outcome  post_year  estimate       se    ci_low   ci_high  p_value     n  clusters
Public primary schools       2020 -0.120324 0.049071 -0.216533 -0.024114 0.014252 18185      3637


## Seminar questions

1. Which measurement scale best answers an absolute service-access question?
2. What assumptions turn an exposure-slope contrast into a causal statement?
3. Why might nearby parishes have correlated errors?
4. What would a pre-period diagnostic establish, and what would it leave unresolved?
5. Why do replication success and source-data validity require separate checks?